# <Method> — E2E (verify before merge)
Publish PRIVATE -> load via trust_remote_code -> run -> validate. Runtime: A100 · HUGGINGFACE_TOKEN · accept FLUX.1-dev.

## 1 · Install + auth

In [ ]:
!pip install -q "git+https://github.com/huggingface/diffusers.git" transformers accelerate sentencepiece protobuf

In [ ]:
import os
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"]="60"
import torch
from huggingface_hub import login
try:
    from google.colab import userdata; login(userdata.get("HUGGINGFACE_TOKEN"))
except Exception:
    login()
DEV,DT="cuda",torch.bfloat16

## 2 · Publish PRIVATE (upload block.py + configs first)

In [ ]:
from huggingface_hub import HfApi
api=HfApi(); REPO="remyxai/<name>-flux-modular"
api.create_repo(REPO, private=True, repo_type="model", exist_ok=True)
for f in ["block.py","modular_config.json","modular_model_index.json"]:
    api.upload_file(path_or_fileobj=f, path_in_repo=f, repo_id=REPO)
print("published PRIVATE:", REPO)

## 3 · Load + run

In [ ]:
from diffusers import ModularPipeline
pipe=ModularPipeline.from_pretrained(REPO, trust_remote_code=True)
print("loaded block:", type(pipe.blocks).__name__)
pipe.load_components(dtype=DT); pipe.to(DEV)
from IPython.display import display
img=pipe(prompt="...").images[0]
display(img)

## 4 · Validate (metric / before-after / no-op control)
Add the quantitative check appropriate to the method (Δ vs reference, CLIP/ArcFace/FID/DreamSim, or feature-off==stock).

## Verdict
Loads as the expected block + produces the claimed result + metric holds -> ready to merge, publish public, link Colab.